# NLP Classification — TF-IDF + Linear Model + Error Analysis

Industrial notebook baseline:
- deterministic dataset generation (multi-topic short texts)
- TF-IDF vectorization
- Linear classifier
- confusion matrix + top errors
- calibration-ish probability inspection

Outputs are saved in the notebook when executed.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

SEED = 1337
rng = np.random.default_rng(SEED)
pd.set_option('display.max_columns', 50)

## 1) Synthetic dataset
We generate short messages with overlapping vocab to make the task non-trivial.

In [2]:
topics = {
    'billing': ['invoice', 'refund', 'price', 'subscription', 'charge', 'payment', 'receipt', 'billing'],
    'technical': ['error', 'crash', 'bug', 'timeout', 'latency', 'api', 'server', 'deploy', 'database'],
    'security': ['phishing', 'malware', 'breach', '2fa', 'token', 'encryption', 'password', 'alert'],
}
common = ['please', 'help', 'cannot', 'issue', 'urgent', 'today', 'account', 'support']

def make_example(label: str) -> str:
    words = []
    words += list(rng.choice(common, size=int(rng.integers(2, 5)), replace=False))
    words += list(rng.choice(topics[label], size=int(rng.integers(3, 6)), replace=False))
    # add some noise from other topics
    other = [k for k in topics.keys() if k != label]
    if rng.random() < 0.35:
        words += list(rng.choice(topics[rng.choice(other)], size=1, replace=False))
    rng.shuffle(words)
    return ' '.join(words)

rows = []
for label in topics.keys():
    for _ in range(900):
        rows.append((make_example(label), label))

df = pd.DataFrame(rows, columns=['text','label'])
df.sample(5, random_state=SEED)

,text,label
1749,bug error urgent issue server api,technical
13,today price help urgent invoice billing,billing
2012,cannot breach today password alert charge account,security
840,today subscription issue urgent charge price p...,billing
2163,refund alert encryption malware support token ...,security


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=SEED, stratify=df['label']
)
X_train.shape, X_test.shape

((2160,), (540,))

## 2) Model

In [4]:
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.98)),
    ('clf', LogisticRegression(max_iter=400, class_weight='balanced')),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

     billing       1.00      1.00      1.00       180
    security       1.00      1.00      1.00       180
   technical       1.00      1.00      1.00       180

    accuracy                           1.00       540
   macro avg       1.00      1.00      1.00       540
weighted avg       1.00      1.00      1.00       540



## 3) Confusion matrix

In [5]:
labels = sorted(df['label'].unique())
cm = confusion_matrix(y_test, pred, labels=labels)
pd.DataFrame(cm, index=[f'true:{l}' for l in labels], columns=[f'pred:{l}' for l in labels])

,pred:billing,pred:security,pred:technical
true:billing,180,0,0
true:security,0,180,0
true:technical,0,0,180


## 4) Top errors (qualitative)

In [6]:
proba = pipe.predict_proba(X_test)
pred_idx = proba.argmax(axis=1)
true_idx = y_test.map({l:i for i,l in enumerate(pipe.classes_) }).values
confidence = proba.max(axis=1)

err = pd.DataFrame({
    'text': X_test.values,
    'true': y_test.values,
    'pred': pred,
    'confidence': confidence,
})
err = err[err['true'] != err['pred']].sort_values('confidence', ascending=False).head(12)
err

,text,true,pred,confidence
